In [1]:
import warnings
import os
warnings.filterwarnings('ignore')
warnings.filterwarnings("ignore", category=UserWarning, module="seqeval")
!pip install -q seqeval
import seqeval
warnings.filterwarnings("ignore", module=r"seqeval\..*")

In [2]:
import os
import logging
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["DISABLE_TQDM"] = "1"
from transformers.utils import logging
logging.set_verbosity_error()
from transformers import logging as transformers_logging
transformers_logging.set_verbosity_error()

In [3]:
!pip install -q transformers datasets seqeval evaluate accelerate --quiet
!pip install -q nltk --quiet

In [4]:
import numpy as np
import torch
import nltk
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    pipeline,
)
import evaluate

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f" Device: {device}")
print(f" PyTorch: {torch.__version__}")

 Device: cuda
 PyTorch: 2.10.0+cu128


In [5]:
nltk.download("conll2000")
nltk.download("punkt")
from nltk.corpus import conll2000
sample_sent = conll2000.iob_sents()[0]
print("Sample sentence (word, POS, chunk):")
for word, pos, chunk in sample_sent[:8]:
    print(f"{word:<15} {pos:<8} {chunk}")

Sample sentence (word, POS, chunk):
Confidence      NN       B-NP
in              IN       B-PP
the             DT       B-NP
pound           NN       I-NP
is              VBZ      B-VP
widely          RB       I-VP
expected        VBN      I-VP
to              TO       I-VP


[nltk_data] Downloading package conll2000 to /root/nltk_data...
[nltk_data]   Package conll2000 is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [6]:
all_sents = conll2000.iob_sents()

all_pos_tags   = sorted(set(pos   for sent in all_sents for _, pos, _     in sent))
all_chunk_tags = sorted(set(chunk for sent in all_sents for _, _,   chunk in sent))

POS_LABEL_LIST   = all_pos_tags
CHUNK_LABEL_LIST = all_chunk_tags

pos_label2id   = {l: i for i, l in enumerate(POS_LABEL_LIST)}
chunk_label2id = {l: i for i, l in enumerate(CHUNK_LABEL_LIST)}

print(f"POS tag classes   ({len(POS_LABEL_LIST)}): {POS_LABEL_LIST}")
print(f"Chunk tag classes ({len(CHUNK_LABEL_LIST)}): {CHUNK_LABEL_LIST}")

POS tag classes   (44): ['#', '$', "''", '(', ')', ',', '.', ':', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB', '``']
Chunk tag classes (23): ['B-ADJP', 'B-ADVP', 'B-CONJP', 'B-INTJ', 'B-LST', 'B-NP', 'B-PP', 'B-PRT', 'B-SBAR', 'B-UCP', 'B-VP', 'I-ADJP', 'I-ADVP', 'I-CONJP', 'I-INTJ', 'I-LST', 'I-NP', 'I-PP', 'I-PRT', 'I-SBAR', 'I-UCP', 'I-VP', 'O']


In [7]:
def nltk_to_records(sents, pos_l2id, chunk_l2id):
    records = []
    for sent in sents:
        tokens     = [w   for w, _, _ in sent]
        pos_ids    = [pos_l2id[p]   for _, p, _ in sent]
        chunk_ids  = [chunk_l2id[c] for _, _, c in sent]
        records.append({
            "tokens"    : tokens,
            "pos_tags"  : pos_ids,
            "chunk_tags": chunk_ids,
        })
    return records

train_sents = conll2000.iob_sents("train.txt")
test_sents  = conll2000.iob_sents("test.txt")

all_train_records = nltk_to_records(train_sents, pos_label2id, chunk_label2id)
test_records      = nltk_to_records(test_sents,  pos_label2id, chunk_label2id)

split_idx    = int(len(all_train_records) * 0.9)
train_records = all_train_records[:split_idx]
val_records   = all_train_records[split_idx:]

print(f"Train   : {len(train_records):,} sentences")
print(f"Val     : {len(val_records):,} sentences")
print(f"Test    : {len(test_records):,} sentences")

Train   : 8,042 sentences
Val     : 894 sentences
Test    : 2,012 sentences


In [8]:
raw_dataset = DatasetDict({
    "train"     : Dataset.from_list(train_records),
    "validation": Dataset.from_list(val_records),
    "test"      : Dataset.from_list(test_records),
})
print(raw_dataset)

ex = raw_dataset["train"][0]
print("\nFirst training sentence:")
print("  tokens    :", ex["tokens"][:8])
print("  pos_tags  :", ex["pos_tags"][:8],  "→", [POS_LABEL_LIST[i]   for i in ex["pos_tags"][:8]])
print("  chunk_tags:", ex["chunk_tags"][:8], "→", [CHUNK_LABEL_LIST[i] for i in ex["chunk_tags"][:8]])

DatasetDict({
    train: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags'],
        num_rows: 8042
    })
    validation: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags'],
        num_rows: 894
    })
    test: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags'],
        num_rows: 2012
    })
})

First training sentence:
  tokens    : ['Confidence', 'in', 'the', 'pound', 'is', 'widely', 'expected', 'to']
  pos_tags  : [18, 13, 10, 18, 38, 26, 36, 31] → ['NN', 'IN', 'DT', 'NN', 'VBZ', 'RB', 'VBN', 'TO']
  chunk_tags: [5, 6, 5, 16, 10, 21, 21, 21] → ['B-NP', 'B-PP', 'B-NP', 'I-NP', 'B-VP', 'I-VP', 'I-VP', 'I-VP']


In [9]:
MODEL_CHECKPOINT = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
TASK         = "pos"
LABEL_LIST   = POS_LABEL_LIST   if TASK == "pos" else CHUNK_LABEL_LIST
LABEL_COLUMN = "pos_tags"       if TASK == "pos" else "chunk_tags"

print(f"Task: {TASK.upper()} Tagging  |  {len(LABEL_LIST)} classes")

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=128,
        padding="max_length",
    )
    aligned_labels = []
    for i, label_ids in enumerate(examples[LABEL_COLUMN]):
        word_ids   = tokenized_inputs.word_ids(batch_index=i)
        prev_word  = None
        label_row  = []
        for word_idx in word_ids:
            if word_idx is None:
                label_row.append(-100)
            elif word_idx != prev_word:
                label_row.append(label_ids[word_idx])
            else:
                label_row.append(-100)
            prev_word = word_idx
        aligned_labels.append(label_row)
        tokenized_inputs["labels"] = aligned_labels
    return tokenized_inputs

tokenized_datasets = raw_dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_dataset["train"].column_names,
)
print("\nTokenization complete!")
sample = tokenized_datasets["train"][0]
print("Fields         :", list(sample.keys()))
print("input_ids len  :", len(sample["input_ids"]))
print("attention_mask :", len(sample["attention_mask"]))
print("labels len     :", len(sample["labels"]))
print("Labels (first 15):", sample["labels"][:15])

Task: POS Tagging  |  44 classes


Map:   0%|          | 0/8042 [00:00<?, ? examples/s]

Map:   0%|          | 0/894 [00:00<?, ? examples/s]

Map:   0%|          | 0/2012 [00:00<?, ? examples/s]


Tokenization complete!
Fields         : ['input_ids', 'token_type_ids', 'attention_mask', 'labels']
input_ids len  : 128
attention_mask : 128
labels len     : 128
Labels (first 15): [-100, 18, 13, 10, 18, 38, 26, 36, 31, 33, 10, 14, 18, 13, 18]


In [10]:
id2label = {i: l for i, l in enumerate(LABEL_LIST)}
label2id = {l: i for i, l in enumerate(LABEL_LIST)}

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(LABEL_LIST),
    id2label=id2label,
    label2id=label2id,
)

print(f"Model : {MODEL_CHECKPOINT}")
print(f"   Params: {model.num_parameters():,}")
print(f"   Head  : Linear({model.config.hidden_size} → {model.config.num_labels})")
print(f"   Labels: {list(id2label.values())}")

Model : distilbert-base-uncased
   Params: 66,396,716
   Head  : Linear(768 → 44)
   Labels: ['#', '$', "''", '(', ')', ',', '.', ':', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB', '``']


In [11]:
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_preds = [
        [LABEL_LIST[pred] for pred, lbl in zip(pred_row, lbl_row) if lbl != -100]
        for pred_row, lbl_row in zip(predictions, labels)
    ]
    true_labels = [
        [LABEL_LIST[lbl] for lbl in lbl_row if lbl != -100]
        for lbl_row in labels
    ]

    results = seqeval.compute(
        predictions=true_preds,
        references=true_labels,
        zero_division=0
    )
    return {
        "precision": results["overall_precision"],
        "recall"   : results["overall_recall"],
        "f1"       : results["overall_f1"],
        "accuracy" : results["overall_accuracy"],
    }

In [12]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
training_args = TrainingArguments(
    output_dir=f"./distilbert-{TASK}-tagging",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=100,
    report_to="none",
    push_to_hub=False,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
print(f"Starting {TASK.upper()} training...")
train_result = trainer.train()
print(f"\nTraining done! Loss: {train_result.training_loss:.4f}")

Starting POS training...
{'loss': '3.339', 'grad_norm': '2.503', 'learning_rate': '1.311e-05', 'epoch': '0.1988'}
{'loss': '0.9827', 'grad_norm': '2.121', 'learning_rate': '1.929e-05', 'epoch': '0.3976'}
{'loss': '0.2813', 'grad_norm': '2.089', 'learning_rate': '1.782e-05', 'epoch': '0.5964'}
{'loss': '0.1979', 'grad_norm': '1.591', 'learning_rate': '1.635e-05', 'epoch': '0.7952'}
{'loss': '0.1571', 'grad_norm': '1.507', 'learning_rate': '1.487e-05', 'epoch': '0.994'}
{'eval_loss': '0.1241', 'eval_precision': '0.9568', 'eval_recall': '0.9562', 'eval_f1': '0.9565', 'eval_accuracy': '0.9683', 'eval_runtime': '5.179', 'eval_samples_per_second': '172.6', 'eval_steps_per_second': '10.81', 'epoch': '1'}
{'loss': '0.1285', 'grad_norm': '1.049', 'learning_rate': '1.34e-05', 'epoch': '1.193'}
{'loss': '0.1158', 'grad_norm': '1.419', 'learning_rate': '1.193e-05', 'epoch': '1.392'}
{'loss': '0.111', 'grad_norm': '0.938', 'learning_rate': '1.046e-05', 'epoch': '1.59'}
{'loss': '0.1027', 'grad_norm

In [13]:
preds, labels, _ = trainer.predict(tokenized_datasets["test"])
preds = np.argmax(preds, axis=2)

true_preds = [
    [LABEL_LIST[p] for p, l in zip(pr, lr) if l != -100]
    for pr, lr in zip(preds, labels)
]
true_labels = [
    [LABEL_LIST[l] for l in lr if l != -100]
    for lr in labels
]
results = seqeval.compute(predictions=true_preds, references=true_labels)
print(f"{'='*48}")
print(f"  {TASK.upper()} TAGGING — Test Set Evaluation")
print(f"{'='*48}")
print(f"  Precision : {results['overall_precision']:.4f}")
print(f"  Recall    : {results['overall_recall']:.4f}")
print(f"  F1 Score  : {results['overall_f1']:.4f}")
print(f"  Accuracy  : {results['overall_accuracy']:.4f}")
print(f"{'='*48}")
print("\nPer-class breakdown:")
for cls, m in results.items():
    if isinstance(m, dict):
        print(f"   {cls:<12} P={m['precision']:.3f}  R={m['recall']:.3f}  F1={m['f1']:.3f}  (n={m['number']})")

  POS TAGGING — Test Set Evaluation
  Precision : 0.9626
  Recall    : 0.9649
  F1 Score  : 0.9637
  Accuracy  : 0.9737

Per-class breakdown:
   '            P=0.997  R=0.997  F1=0.997  (n=313)
   B            P=0.956  R=0.963  F1=0.959  (n=2322)
   BD           P=0.958  R=0.973  F1=0.965  (n=1678)
   BG           P=0.940  R=0.971  F1=0.955  (n=724)
   BN           P=0.932  R=0.926  F1=0.929  (n=1073)
   BP           P=0.947  R=0.935  F1=0.941  (n=539)
   BR           P=0.921  R=0.817  F1=0.866  (n=71)
   BS           P=0.889  R=0.980  F1=0.932  (n=49)
   BZ           P=0.980  R=0.968  F1=0.974  (n=913)
   C            P=1.000  R=0.997  F1=0.998  (n=1214)
   D            P=0.981  R=0.994  F1=0.988  (n=1972)
   DT           P=0.971  R=0.934  F1=0.952  (n=212)
   H            P=0.000  R=0.000  F1=0.000  (n=2)
   J            P=0.910  R=0.939  F1=0.924  (n=2742)
   JR           P=0.916  R=0.975  F1=0.945  (n=201)
   JS           P=0.973  R=0.922  F1=0.947  (n=77)
   N            P=0.952  

In [14]:
tag_pipeline = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="first",
    device=0 if device == "cuda" else -1,
)
def predict_tags(sentence):
    out = tag_pipeline(sentence)
    print(f"\n'{sentence}'")
    print(f"{'Word':<22} {'Tag':<12} Score")
    print("-" * 44)
    for item in out:
        word = item["word"].lstrip("##")
        print(f"{word:<22} {item['entity_group']:<12} {item['score']:.3f}")

predict_tags("Bran works at Google in California")
predict_tags("The quick brown cheetah jumps over the lazy goat")
predict_tags("Microsoft released a new software last Monday")


'Bran works at Google in California'
Word                   Tag          Score
--------------------------------------------
bran                   NNP          0.986
works                  VBZ          0.555
at                     IN           0.994
google                 NNP          0.992
in                     IN           0.992
california             NNP          0.990

'The quick brown cheetah jumps over the lazy goat'
Word                   Tag          Score
--------------------------------------------
the                    DT           0.996
quick                  JJ           0.486
brown cheetah          NNP          0.640
jumps                  VBZ          0.828
over                   IN           0.996
the                    DT           0.996
lazy                   JJ           0.951
goat                   NN           0.978

'Microsoft released a new software last Monday'
Word                   Tag          Score
--------------------------------------------
microsoft   

In [ ]:
def run_task(task_name):
    label_list = POS_LABEL_LIST if task_name == "pos" else CHUNK_LABEL_LIST
    label_col  = "pos_tags"     if task_name == "pos" else "chunk_tags"

    def _align(examples):
        tok = tokenizer(
            examples["tokens"], truncation=True, is_split_into_words=True,
            max_length=128, padding="max_length",
        )
        aligned = []
        for i, lbl_ids in enumerate(examples[label_col]):
            word_ids = tok.word_ids(batch_index=i)
            prev, row = None, []
            for wid in word_ids:
                if wid is None:       row.append(-100)
                elif wid != prev:     row.append(lbl_ids[wid])
                else:                 row.append(-100)
                prev = wid
            aligned.append(row)
        tok["labels"] = aligned
        return tok

    ds = raw_dataset.map(_align, batched=True,
                         remove_columns=raw_dataset["train"].column_names)

    _id2label = {i: l for i, l in enumerate(label_list)}
    _label2id = {l: i for i, l in enumerate(label_list)}
    _model = AutoModelForTokenClassification.from_pretrained(
        MODEL_CHECKPOINT, num_labels=len(label_list),
        id2label=_id2label, label2id=_label2id,
    ).to(device)

    def _metrics(p):
        pr, lb = p
        pr = np.argmax(pr, axis=2)
        tp = [[label_list[x] for x, y in zip(r, s) if y != -100] for r, s in zip(pr, lb)]
        tl = [[label_list[y] for y in s if y != -100] for s in lb]
        r  = seqeval.compute(predictions=tp, references=tl)
        return {"precision": r["overall_precision"], "recall": r["overall_recall"],
                "f1": r["overall_f1"], "accuracy": r["overall_accuracy"]}

    _args = TrainingArguments(
        output_dir=f"./distilbert-{task_name}", num_train_epochs=3,
        per_device_train_batch_size=16, per_device_eval_batch_size=16,
        learning_rate=2e-5, weight_decay=0.01, eval_strategy="epoch",
        save_strategy="epoch", load_best_model_at_end=True,
        logging_steps=200, report_to="none", push_to_hub=False,
    )
    _trainer = Trainer(
        model=_model, args=_args,
        train_dataset=ds["train"], eval_dataset=ds["validation"],
        data_collator=DataCollatorForTokenClassification(tokenizer),
        compute_metrics=_metrics,
    )
    _trainer.train()
    preds, lbls, _ = _trainer.predict(ds["test"])
    preds = np.argmax(preds, axis=2)
    tp = [[label_list[p] for p, l in zip(pr, lr) if l != -100] for pr, lr in zip(preds, lbls)]
    tl = [[label_list[l] for l in lr if l != -100] for lr in lbls]
    metrics = seqeval.compute(predictions=tp, references=tl)

    _model.to("cpu")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    del _model
    del _trainer
    import gc
    gc.collect()

    return metrics


print("Training POS model...")
pos_res = run_task("pos")

print("\nTraining Chunking model...")
chunk_res = run_task("chunk")
print(f"\n{'='*55}")
print(f"{'Metric':<15} {'POS Tagging':>18} {'Chunking':>18}")
print(f"{'='*55}")
for key in ["overall_precision", "overall_recall", "overall_f1", "overall_accuracy"]:
    label = key.replace("overall_", "").capitalize()
    print(f"{label:<15} {pos_res[key]:>18.4f} {chunk_res[key]:>18.4f}")
print(f"{'='*55}")

Training POS model...


Map:   0%|          | 0/8042 [00:00<?, ? examples/s]

Map:   0%|          | 0/894 [00:00<?, ? examples/s]

Map:   0%|          | 0/2012 [00:00<?, ? examples/s]

{'loss': '1.17', 'grad_norm': '1.399', 'learning_rate': '1.736e-05', 'epoch': '0.3976'}
{'loss': '0.2083', 'grad_norm': '1.536', 'learning_rate': '1.471e-05', 'epoch': '0.7952'}
{'eval_loss': '0.1216', 'eval_precision': '0.9578', 'eval_recall': '0.9568', 'eval_f1': '0.9573', 'eval_accuracy': '0.9694', 'eval_runtime': '5.182', 'eval_samples_per_second': '172.5', 'eval_steps_per_second': '10.81', 'epoch': '1'}
{'loss': '0.1379', 'grad_norm': '1.136', 'learning_rate': '1.206e-05', 'epoch': '1.193'}
{'loss': '0.1136', 'grad_norm': '1.009', 'learning_rate': '9.41e-06', 'epoch': '1.59'}
{'loss': '0.1022', 'grad_norm': '1.069', 'learning_rate': '6.759e-06', 'epoch': '1.988'}
{'eval_loss': '0.08921', 'eval_precision': '0.9664', 'eval_recall': '0.9669', 'eval_f1': '0.9667', 'eval_accuracy': '0.9764', 'eval_runtime': '4.523', 'eval_samples_per_second': '197.7', 'eval_steps_per_second': '12.38', 'epoch': '2'}
